# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/M-Abdullah-Nazeer/Flyrank-Ml-Pipeline/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!pip -q install duckdb
import duckdb, re, os
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from google.colab import userdata
from huggingface_hub import HfApi

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")
rel = "hf://datasets/FlyRank/internship-warehouse"
files = HfApi().list_repo_files("FlyRank/internship-warehouse", repo_type="dataset", token=userdata.get("HF_TOKEN"))
print(sorted({m.group(1) for f in files if (m := re.search(r"month=(\d{4}-\d{2})", f))}))

['2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06', '2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12', '2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2026-06']


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Task:** predict which visible pages will keep under-capturing clicks for their position (label = 1 if days 16-end clicks are below 50% of what their position bucket predicts), so a content team knows where to review titles and metadata first.
**Methods:** (a) the Week-4 baseline, missed clicks = impressions × (expected CTR − actual CTR); (b) logistic regression, a simple, readable model; (c) gradient boosting, to see if non-linear patterns add anything. Gradient boosting is safe here because the features only use days 1-15 and the split respects time and clients.
**Rule for choosing:** I only prefer the more complex model if it beats the simpler ones on the held-out month by a clear margin. Complexity alone is not a reason.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Time-aware split:** train on 2026-02 and 2026-03, test on 2026-04. The model never sees a later month while training, matching the real use (score pages now, see what happens next).
**Inside each month:** features come from days 1-15 only, the label from days 16-end. The expected-CTR curve is computed from the training months only.
**Client check:** as a second view, I hold out whole clients inside the training months, so the model is not just remembering one client's pages.
**Same as Week 4:** the baseline is scored on exactly the same test pages with the same metrics. The sealed final month (2026-06) is not used.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
def build_month(m):
    con.sql(f"""CREATE OR REPLACE TABLE dm AS
      SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/month={m}/*.parquet')""")
    df = con.sql("""
    WITH past AS (
      SELECT client_hash_id, content_hash_id,
             SUM(gsc_impressions) AS impressions, SUM(gsc_clicks) AS clicks,
             SUM(gsc_avg_position * gsc_impressions) / NULLIF(SUM(gsc_impressions), 0) AS position,
             COUNT(*) AS days_with_data,
             SUM(CASE WHEN day(report_date) BETWEEN 8 AND 15 THEN gsc_impressions END) AS impr_late,
             SUM(CASE WHEN day(report_date) BETWEEN 1 AND 7 THEN gsc_impressions END) AS impr_early
      FROM dm WHERE gsc_data_available IS TRUE AND gsc_impressions > 0 AND day(report_date) <= 15
      GROUP BY 1, 2),
    fut AS (
      SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS f_impr, SUM(gsc_clicks) AS f_clicks
      FROM dm WHERE gsc_data_available IS TRUE AND day(report_date) > 15
      GROUP BY 1, 2)
    SELECT p.*, f.f_impr, f.f_clicks FROM past p JOIN fut f USING (client_hash_id, content_hash_id)
    """).df()
    df["month"] = m
    return df

TRAIN_MONTHS, TEST_MONTH = ["2026-02", "2026-03"], "2026-04"
assert "2026-06" not in TRAIN_MONTHS + [TEST_MONTH]
raw = {m: build_month(m) for m in TRAIN_MONTHS + [TEST_MONTH]}

def eligible(df):
    return df[(df["impressions"] >= 100) & (df["position"] <= 20) & (df["f_impr"] >= 100)].copy()

train = eligible(pd.concat([raw[m] for m in TRAIN_MONTHS], ignore_index=True))
test = eligible(raw[TEST_MONTH])

bins, labels = [0, 3, 6, 10, 20], ["1-3", "4-6", "7-10", "11-20"]
tb = pd.cut(train["position"], bins, labels=labels)
expected = (train.groupby(tb, observed=True)["clicks"].sum() /
            train.groupby(tb, observed=True)["impressions"].sum()).to_dict()   # training months only

def add_cols(df):
    df["pos_bucket"] = pd.cut(df["position"], bins, labels=labels)
    df["expected_ctr"] = df["pos_bucket"].map(expected).astype(float)
    df["ctr"] = df["clicks"] / df["impressions"]
    df["log_impr"] = np.log10(df["impressions"])
    df["ctr_gap_ratio"] = (df["ctr"] / df["expected_ctr"]).clip(upper=10)
    df["trend_ratio"] = (df["impr_late"] / df["impr_early"]).replace([np.inf, -np.inf], np.nan).clip(upper=10)
    df["missed_clicks"] = df["impressions"] * (df["expected_ctr"] - df["ctr"])      # the Week-4 score
    df["label"] = (df["f_clicks"] < 0.5 * df["f_impr"] * df["expected_ctr"]).astype(int)
    return df

train, test = add_cols(train), add_cols(test)
print("train", train.shape, "label rate", round(train["label"].mean(), 3))
print("test ", test.shape, "label rate", round(test["label"].mean(), 3))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

train (105942, 19) label rate 0.515
test  (59986, 19) label rate 0.497


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

FEATURES = ["log_impr", "position", "ctr", "ctr_gap_ratio", "trend_ratio", "days_with_data"]

logreg = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                       LogisticRegression(max_iter=1000)).fit(train[FEATURES], train["label"])
gboost = HistGradientBoostingClassifier(random_state=0).fit(train[FEATURES], train["label"])

test["s_baseline"] = test["missed_clicks"]                       # Week-4 score
test["s_gap_only"] = -test["ctr_gap_ratio"]
test["s_logreg"] = logreg.predict_proba(test[FEATURES])[:, 1]
test["s_gboost"] = gboost.predict_proba(test[FEATURES])[:, 1]

def prec_at_k(score, y, k):
    idx = np.argsort(-np.asarray(score))[:k]
    return float(np.asarray(y)[idx].mean())

y = test["label"].values
rows = []
for name, col in [("Week-4 baseline (missed clicks)", "s_baseline"), ("CTR gap ratio alone", "s_gap_only"),
                  ("Logistic regression", "s_logreg"), ("Gradient boosting", "s_gboost")]:
    rows.append({"method": name, "AUC": round(roc_auc_score(y, test[col]), 3),
                 "precision@100": round(prec_at_k(test[col], y, 100), 3),
                 "precision@500": round(prec_at_k(test[col], y, 500), 3),
                 "precision@1000": round(prec_at_k(test[col], y, 1000), 3)})
results = pd.DataFrame(rows)
print(f"test month {TEST_MONTH}: n = {len(test)}, base rate = {y.mean():.3f}")
display(results)

os.makedirs("work/outputs", exist_ok=True)
results.to_csv("work/outputs/w05_model_vs_baseline.csv", index=False)

# Client-grouped check inside the training months
from sklearn.model_selection import GroupShuffleSplit
from sklearn.base import clone

tr, te = next(GroupShuffleSplit(test_size=0.25, random_state=0).split(train, groups=train["client_hash_id"]))
for name, mdl in [("logistic regression", logreg), ("gradient boosting", gboost)]:
    m = clone(mdl).fit(train.iloc[tr][FEATURES], train.iloc[tr]["label"])
    print(name, "AUC on held-out clients:",
          round(roc_auc_score(train.iloc[te]["label"], m.predict_proba(train.iloc[te][FEATURES])[:, 1]), 3))

test month 2026-04: n = 59986, base rate = 0.497


,method,AUC,precision@100,precision@500,precision@1000
0,Week-4 baseline (missed clicks),0.680,0.79,0.786,0.763
1,CTR gap ratio alone,0.745,0.78,0.712,0.735
2,Logistic regression,0.750,0.87,0.824,0.803
3,Gradient boosting,0.789,0.90,0.918,0.911


logistic regression AUC on held-out clients: 0.699
gradient boosting AUC on held-out clients: 0.746


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

**What the model leans on:** permutation importance (AUC drop when a feature is shuffled) is dominated by `ctr_gap_ratio` (0.177), far ahead of `log_impr` (0.044) and `ctr` (0.015). `days_with_data`, `position` and `trend_ratio` add almost nothing (each below 0.006). This agrees with the signal audit: the main warning sign is a CTR that is low for the page's position. In logistic regression, the coefficients point the same way: a higher CTR (-0.83) and a higher CTR gap ratio (-0.32) lower the risk, and higher volume also lowers it (`log_impr` -0.39), meaning that among pages with a similar CTR, bigger pages are less likely to keep under-capturing. `ctr` and `ctr_gap_ratio` are strongly related (the gap ratio is CTR divided by the expected CTR), so I read those two coefficients together and not one by one.

**Where it is wrong:** 8.9% of the model's top 1,000 picks are wrong (precision@1000 = 0.911). Wrong rates by position bucket: 1-3 is 13.5% (n = 304), 4-6 is 7.7% (n = 388), 7-10 is 7.0% (n = 228), 11-20 is 2.5% (n = 80). So the errors concentrate on top-3 pages, where a search feature or a branded query may take the click for reasons a title rewrite cannot change. By volume, 1k-5k pages are wrong 13.1% of the time (n = 222) versus 7.7% for 5k+ (n = 778); with fewer impressions, a page's future CTR is noisier. The top picks are all pages with 1,000+ impressions (78% have 5,000+).

**What the misses look like:** the pages with label 1 that the model ranked lowest have the same median `days_with_data` as its top picks (15 vs 15), so missing data days does not explain the misses. I did not find a simple trait that separates them using the features here, which suggests the model's blind spots depend on things not in the data, such as the query mix or search features.

**Validation and honest framing:** the time-aware test (train 2026-02/03, test 2026-04) gives AUC 0.789 for gradient boosting. When whole clients are held out inside the training months, AUC drops to 0.746 (logistic regression 0.699), so part of the April performance probably comes from clients the model had already seen. I did not compute the Week-4 baseline on those held-out clients, so I cannot claim the model beats the baseline for brand-new clients. These are observed, directional results from three months of data, useful for decision-support (which pages to review first), not proof of why a page under-captures or that a rewrite would fix it.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.inspection import permutation_importance

# What the best model leans on (permutation importance on the test month)
sample = test.sample(min(20000, len(test)), random_state=0)
pi = permutation_importance(gboost, sample[FEATURES], sample["label"], scoring="roc_auc",
                            n_repeats=3, random_state=0)
imp = pd.Series(pi.importances_mean, index=FEATURES).sort_values(ascending=False).round(4)
display(imp.to_frame("AUC drop when shuffled"))

# Logistic regression direction of each feature (standardized)
coef = pd.Series(logreg[-1].coef_[0], index=FEATURES).round(3)
display(coef.to_frame("logreg coefficient"))

# Where is the model wrong? Look at its top 1000 picks
best = "s_gboost"            # change to "s_logreg" if that one won
top = test.sort_values(best, ascending=False).head(1000).copy()
top["wrong"] = (top["label"] == 0)
print("share of top-1000 picks that are wrong:", round(top["wrong"].mean(), 3))
display(top.groupby("pos_bucket", observed=True)["wrong"].agg(["size", "mean"]).round(3).rename(columns={"size": "n", "mean": "wrong_rate"}))
top["vol_bucket"] = pd.cut(top["impressions"], [0, 300, 1000, 5000, 1e12], labels=["100-300", "300-1k", "1k-5k", "5k+"])
display(top.groupby("vol_bucket", observed=True)["wrong"].agg(["size", "mean"]).round(3).rename(columns={"size": "n", "mean": "wrong_rate"}))

# What does a miss look like? (label 1 pages the model ranked low)
miss = test[(test["label"] == 1)].sort_values(best).head(1000)
print("median days_with_data, missed vs top picks:", miss["days_with_data"].median(), top["days_with_data"].median())

,AUC drop when shuffled
ctr_gap_ratio,0.1769
log_impr,0.0442
ctr,0.0154
days_with_data,0.0058
position,0.0045
trend_ratio,0.0043


,logreg coefficient
log_impr,-0.387
position,0.040
ctr,-0.828
ctr_gap_ratio,-0.322
trend_ratio,0.032
days_with_data,0.130


share of top-1000 picks that are wrong: 0.089


,n,wrong_rate
pos_bucket,,
1-3,304,0.135
4-6,388,0.077
7-10,228,0.070
11-20,80,0.025


,n,wrong_rate
vol_bucket,,
1k-5k,222,0.131
5k+,778,0.077


median days_with_data, missed vs top picks: 15.0 15.0


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.